In [1]:
# set up environment
import sys
import json
import re
import requests
import pandas as pd
from pathlib import Path
import json
import csv
from caveclient import CAVEclient

import urllib.parse

from pathlib import Path
#pip install neuprint-python
from neuprint import Client, fetch_neurons, fetch_custom, NeuronCriteria as NC


print("Python executable:", sys.executable)
print("Imports OK")

c:\Users\JHS\Documents\Python\cave_env\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Python executable: c:\Users\JHS\Documents\Python\cave_env\Scripts\python.exe
Imports OK


In [2]:
# set up directories
PROJECT_ROOT = Path.cwd() #anchor to current notebook location


OUTPUT_DIR = PROJECT_ROOT / "output-MANCv1.2.3_04B_add neuromeres T1R 20260413"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


print("Output:", OUTPUT_DIR)



Output: c:\Users\JHS\Documents\Python\project_folder_4B\output-MANCv1.2.3_04B_add neuromeres T1R 20260413


In [4]:
from dotenv import load_dotenv
import os

load_dotenv()  # loads .env from current working directory

token = os.environ["NEUPRINT_APPLICATION_CREDENTIALS"]

In [5]:
from neuprint import Client

c = Client(
    "https://neuprint.janelia.org",
    dataset="manc:v1.2.3",
    token=token
)
print(c)

Client("https://neuprint.janelia.org", "manc:v1.2.3")


In [6]:
#Fetch neurons to populate the state later: one neuromere with 04B add T1 right side
from neuprint import fetch_custom

cypher = """
MATCH (n:Neuron)
WHERE n.hemilineage = '04B'
  AND n.somaNeuromere = 'T1'
  AND n.somaSide = 'RHS'
RETURN
  n.bodyId AS bodyId,
  n.type AS type,
  n.instance AS instance,
  n.class AS class,
  n.subclass AS subclass,
  n.hemilineage AS hemilineage,
  n.somaNeuromere AS somaNeuromere,
  n.somaSide AS somaSide,
  n.rootSide AS rootSide,
  n.longTract AS longTract,
  n.birthtime AS birthtime
ORDER BY bodyId
"""

df_4b_t1_rhs = fetch_custom(cypher, client=c)
df_4b_t1_rhs.head() 
df_4b_t1_rhs["bodyId"].nunique() # n=97

98

In [7]:
# Generate a new state that diaplays the 4B morphology clusters, as layers for each subclass WITH  all neurons in one layer displayed with the same color. 04BT2l

import json
import urllib.parse
from itertools import cycle


#generate new, clean state:

##define sources:
# 1) Paste a known-good MANC Clio URL here (we wont repopulate it; it is jsut for retrieving sources)
BASE_URL = "https://clio-ng.janelia.org/#!%7B%22title%22:%22manc-v1.2.3-neuprint-layers%22%2C%22dimensions%22:%7B%22x%22:%5B8e-9%2C%22m%22%5D%2C%22y%22:%5B8e-9%2C%22m%22%5D%2C%22z%22:%5B8e-9%2C%22m%22%5D%7D%2C%22position%22:%5B23056.5%2C29733.5%2C41138.5%5D%2C%22crossSectionOrientation%22:%5B0%2C0.7071067690849304%2C-0.7071067690849304%2C0%5D%2C%22crossSectionScale%22:1%2C%22projectionOrientation%22:%5B0%2C0.7071067690849304%2C-0.7071067690849304%2C0%5D%2C%22projectionScale%22:91364.04452716278%2C%22layers%22:%5B%7B%22type%22:%22image%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-2-26-213dba213ef26e094c16c860ae7f4be0/v3_emdata_clahe_xy/jpeg%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22tab%22:%22rendering%22%2C%22name%22:%22em%22%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22toolBindings%22:%7B%22Q%22:%22selectSegments%22%7D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2210007%22%5D%2C%22segmentColors%22:%7B%2233946%22:%22#808080%22%7D%2C%22name%22:%22manc:v1.2.3%22%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-roi-d5f392696f7a48e27f49fa1a9db5ee3b/all-vnc-roi%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22pick%22:false%2C%22tab%22:%22rendering%22%2C%22selectedAlpha%22:0%2C%22saturation%22:0%2C%22meshSilhouetteRendering%22:4%2C%22segments%22:%5B%221%22%5D%2C%22segmentDefaultColor%22:%22#ffffff%22%2C%22name%22:%22all-tissue%22%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-roi-d5f392696f7a48e27f49fa1a9db5ee3b/synaptic-neuropil%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22pick%22:false%2C%22tab%22:%22rendering%22%2C%22selectedAlpha%22:0%2C%22saturation%22:0%2C%22meshSilhouetteRendering%22:4%2C%22segments%22:%5B%221%22%5D%2C%22segmentDefaultColor%22:%22#ffffff%22%2C%22segmentColors%22:%7B%221%22:%22#ffffff%22%7D%2C%22name%22:%22all-synaptic%22%2C%22archived%22:true%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-roi-d5f392696f7a48e27f49fa1a9db5ee3b/roi-202208%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22pick%22:false%2C%22tab%22:%22segments%22%2C%22selectedAlpha%22:0%2C%22saturation%22:0.5%2C%22meshSilhouetteRendering%22:4%2C%22segments%22:%5B%2210%22%2C%2211%22%2C%2212%22%2C%2213%22%2C%2214%22%2C%2215%22%2C%2216%22%2C%2217%22%2C%2218%22%2C%2219%22%2C%2220%22%2C%2221%22%2C%2222%22%2C%2223%22%2C%2224%22%2C%2225%22%2C%2226%22%2C%2227%22%2C%224%22%2C%225%22%2C%226%22%2C%227%22%2C%228%22%2C%229%22%5D%2C%22name%22:%22neuropils%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-roi-d5f392696f7a48e27f49fa1a9db5ee3b/nerve-roi-202301%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22pick%22:false%2C%22tab%22:%22segments%22%2C%22objectAlpha%22:0.1%2C%22segments%22:%5B%221%22%2C%2210%22%2C%2211%22%2C%2212%22%2C%2213%22%2C%2214%22%2C%2215%22%2C%2216%22%2C%2217%22%2C%2218%22%2C%2219%22%2C%222%22%2C%2220%22%2C%2221%22%2C%2222%22%2C%2223%22%2C%2224%22%2C%2225%22%2C%2226%22%2C%2227%22%2C%2228%22%2C%2229%22%2C%223%22%2C%2230%22%2C%2231%22%2C%2232%22%2C%2233%22%2C%2234%22%2C%2235%22%2C%224%22%2C%225%22%2C%226%22%2C%227%22%2C%228%22%2C%229%22%5D%2C%22name%22:%22nerves%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22annotation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-v1.2-synapse-partners-minconf-0.0.precomputed%22%2C%22tab%22:%22rendering%22%2C%22ignoreNullSegmentFilter%22:false%2C%22shader%22:%22#uicontrol%20bool%20showPre%20checkbox%28default=true%29%5Cn#uicontrol%20bool%20showPostPartners%20checkbox%28default=true%29%5Cn%5Cn#uicontrol%20vec3%20preColor%20color%28default=%5C%22red%5C%22%29%5Cn#uicontrol%20vec3%20postColor%20color%28default=%5C%22blue%5C%22%29%5Cn%5Cn//%20Note:%5Cn//%20%20%20For%20the%20MANC%20in%20neuprint%2C%5Cn//%20%20%20connection%20strengths%20don%27t%5Cn//%20%20%20include%20any%20synapses%20below%5Cn//%20%20%20confidence%200.4.%5Cn#uicontrol%20float%20preConfidence%20slider%28min=0%2C%20max=1%2C%20default=0%29%5Cn#uicontrol%20float%20postConfidence%20slider%28min=0%2C%20max=1%2C%20default=0%29%5Cn%5Cnvoid%20main%28%29%20%7B%5Cn%20%20setColor%28defaultColor%28%29%29%3B%5Cn%5Cn%20%20float%20preSize%20=%206.0%3B%5Cn%20%20float%20postSize%20=%204.0%3B%5Cn%20%20%5Cn%20%20setEndpointMarkerColor%28%5Cn%20%20%20%20vec4%28preColor%2C%200.5%29%2C%5Cn%20%20%20%20vec4%28postColor%2C%200.5%29%29%3B%5Cn%20%20%5Cn%20%20if%20%28showPre%20&&%20showPostPartners%29%20%7B%5Cn%20%20%20%20setLineWidth%281.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%28preSize%2C%20postSize%29%3B%5Cn%20%20%7D%5Cn%20%20else%20if%20%28showPostPartners%29%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%280.0%2C%20postSize%29%3B%5Cn%20%20%7D%5Cn%20%20else%20if%20%28showPre%29%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%28preSize%2C%200.0%29%3B%5Cn%20%20%7D%5Cn%20%20else%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%280.0%2C%200.0%29%3B%5Cn%20%20%7D%5Cn%20%20%5Cn%20%20if%20%28prop_pre_synaptic_confidence%28%29%20%3C%20preConfidence%20%7C%7C%5Cn%20%20%20%20%20%20prop_post_synaptic_confidence%28%29%20%3C%20postConfidence%29%20discard%3B%5Cn%7D%5Cn%22%2C%22shaderControls%22:%7B%22showPostPartners%22:false%2C%22postColor%22:%22#00ffff%22%2C%22preConfidence%22:0.4%2C%22postConfidence%22:0.4%7D%2C%22linkedSegmentationLayer%22:%7B%22pre_synaptic_cell%22:%22manc:v1.2.3%22%7D%2C%22filterBySegmentation%22:%5B%22pre_synaptic_cell%22%5D%2C%22name%22:%22presyn%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22annotation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-v1.2-synapse-partners-minconf-0.0.precomputed%22%2C%22tab%22:%22rendering%22%2C%22ignoreNullSegmentFilter%22:false%2C%22shader%22:%22#uicontrol%20bool%20showPrePartners%20checkbox%28default=true%29%5Cn#uicontrol%20bool%20showPost%20checkbox%28default=true%29%5Cn%5Cn#uicontrol%20vec3%20preColor%20color%28default=%5C%22red%5C%22%29%5Cn#uicontrol%20vec3%20postColor%20color%28default=%5C%22blue%5C%22%29%5Cn%5Cn//%20Note:%5Cn//%20%20%20For%20the%20MANC%20in%20neuprint%2C%5Cn//%20%20%20connection%20strengths%20don%27t%5Cn//%20%20%20include%20any%20synapses%20below%5Cn//%20%20%20confidence%200.4.%5Cn#uicontrol%20float%20preConfidence%20slider%28min=0%2C%20max=1%2C%20default=0.4%29%5Cn#uicontrol%20float%20postConfidence%20slider%28min=0%2C%20max=1%2C%20default=0.4%29%5Cn%5Cnvoid%20main%28%29%20%7B%5Cn%20%20setColor%28defaultColor%28%29%29%3B%5Cn%5Cn%20%20float%20preSize%20=%206.0%3B%5Cn%20%20float%20postSize%20=%204.0%3B%5Cn%20%20%5Cn%20%20setEndpointMarkerColor%28%5Cn%20%20%20%20vec4%28preColor%2C%200.5%29%2C%5Cn%20%20%20%20vec4%28postColor%2C%200.5%29%29%3B%5Cn%20%20%5Cn%20%20if%20%28showPrePartners%20&&%20showPost%29%20%7B%5Cn%20%20%20%20setLineWidth%281.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%28preSize%2C%20postSize%29%3B%5Cn%20%20%7D%5Cn%20%20else%20if%20%28showPost%29%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%280.0%2C%20postSize%29%3B%5Cn%20%20%7D%5Cn%20%20else%20if%20%28showPrePartners%29%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%28preSize%2C%200.0%29%3B%5Cn%20%20%7D%5Cn%20%20else%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%280.0%2C%200.0%29%3B%5Cn%20%20%7D%5Cn%20%20%5Cn%20%20if%20%28prop_pre_synaptic_confidence%28%29%20%3C%20preConfidence%20%7C%7C%5Cn%20%20%20%20%20%20prop_post_synaptic_confidence%28%29%20%3C%20postConfidence%29%20discard%3B%5Cn%7D%5Cn%22%2C%22shaderControls%22:%7B%22showPrePartners%22:false%2C%22postColor%22:%22#00ffff%22%7D%2C%22linkedSegmentationLayer%22:%7B%22post_synaptic_cell%22:%22manc:v1.2.3%22%7D%2C%22filterBySegmentation%22:%5B%22post_synaptic_cell%22%5D%2C%22name%22:%22postsyn%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://vnc-v3-seg-3d2f1c08fd4720848061f77362dc6c17/mask%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22pick%22:false%2C%22tab%22:%22segments%22%2C%22segmentColors%22:%7B%221%22:%22#000000%22%7D%2C%22name%22:%22voxel-classes%22%2C%22archived%22:true%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://vnc-v3-seg-3d2f1c08fd4720848061f77362dc6c17/rc5_wsexp%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22tab%22:%22segments%22%2C%22name%22:%22initial-supervoxels%22%2C%22archived%22:true%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-roi-d5f392696f7a48e27f49fa1a9db5ee3b/court-et-al-systematic-manc_tracts/mesh%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22tab%22:%22rendering%22%2C%22saturation%22:0.5%2C%22meshSilhouetteRendering%22:4%2C%22segments%22:%5B%22100%22%2C%22101%22%2C%22102%22%2C%22103%22%2C%22104%22%2C%22105%22%2C%22106%22%5D%2C%22name%22:%22court%20et%20al.%20tracts%22%2C%22archived%22:true%7D%5D%2C%22showSlices%22:false%2C%22prefetch%22:false%2C%22selectedLayer%22:%7B%22flex%22:1.49%2C%22size%22:426%2C%22visible%22:true%2C%22layer%22:%22manc:v1.2.3%22%7D%2C%22projectionBackgroundColor%22:%22#ffffff%22%2C%22layout%22:%223d%22%2C%22settingsPanel%22:%7B%22visible%22:true%7D%2C%22selection%22:%7B%22flex%22:0.51%2C%22size%22:426%2C%22visible%22:false%7D%7D"

# 2) Decode its state
encoded_state = BASE_URL.split("#!", 1)[1]
base_state = json.loads(urllib.parse.unquote(encoded_state))

# 3) Extract the actual working sources from that scene
em_source = None
seg_source = None

for L in base_state["layers"]:
    if L.get("type") == "image" and em_source is None:
        em_source = L.get("source")
    if L.get("type") == "segmentation" and seg_source is None:
        seg_source = L.get("source")

print("EM source:", em_source)
print("Seg source:", seg_source)

##generate state
new_state = {
    "title": "MANC_v1.2.3_4B_T1r",
    "dimensions": base_state["dimensions"],
    "position": base_state["position"],
    "crossSectionOrientation": base_state["crossSectionOrientation"],
    "crossSectionScale": base_state["crossSectionScale"],
    "projectionOrientation": base_state["projectionOrientation"],
    "projectionScale": base_state["projectionScale"],
    "layout": base_state.get("layout", "3d"),
    "showSlices": base_state.get("showSlices", True),
    "projectionBackgroundColor": base_state.get("projectionBackgroundColor", "#ffffff"),
    "crossSectionBackgroundColor": base_state.get("crossSectionBackgroundColor", "#ffffff"),
    "selectedLayer": base_state.get("selectedLayer"),
    "settingsPanel": base_state.get("settingsPanel", {"visible": True}),
    "layers": [
        {
            "type": "image",
            "name": "EM",
            "source": em_source,
            "visible": True
        }
    ]
}


# --- Color palette (cycles if needed) ---
COLORS = [
    "#87CEEB",  # sky blue
    "#FFA500",  # orange
    "#E34234",  # vermillion
    "#CC99FF",  # pale violet
    "#66CDAA",  # medium aquamarine
    "#FFD700",  # gold
]

color_cycle = cycle(COLORS)


# --- Helper: upsert layer (FIXED) ---
def add_segmentation_layer(state, name, body_ids, seg_source, color):
    body_ids = sorted(set(int(x) for x in body_ids))

    layer = {
        "type": "segmentation",
        "name": name,
        "source": seg_source,
        "segments": [str(x) for x in body_ids],
        "segmentColors": {str(x): color for x in body_ids},
        "visible": True,
    }

    # replace if exists
    for i, existing in enumerate(state["layers"]):
        if existing.get("name") == name:
            state["layers"][i] = layer
            return

    # otherwise append
    state["layers"].append(layer)


# --- Clean subclass labels ---
subclass_df = df_4b_t1_rhs.copy()
subclass_df["subclass"] = (
    subclass_df["subclass"]
    .fillna("unclassified")
    .astype(str)
    .str.strip()
)
subclass_df.loc[subclass_df["subclass"] == "", "subclass"] = "unclassified"


# --- Inspect counts ---
subclass_counts = (
    subclass_df.groupby("subclass")["bodyId"]
    .nunique()
    .sort_values(ascending=False)
)
print(subclass_counts)


# --- Build layers with colors ---
for subclass_name, group in subclass_df.groupby("subclass", sort=True):
    body_ids = group["bodyId"].dropna().astype(int).tolist()
    if not body_ids:
        continue

    layer_name = f"MANC_v1.2.3_{subclass_name}"
    color = next(color_cycle)

    add_segmentation_layer(new_state, layer_name, body_ids, seg_source, color)


#add more manual layers based on my cluster review. 
#use a dictionary and loop to add the layers:
##these came from notebook 4
ir_2_04B_1_ids = []
ir_2_04B_2_ids = [] 
ir_2_04B_3_ids = [] 
ir_2_04B_4_ids =[] 
ir_2_04B_5_ids = []
ir_2_04B_6_ids = []

ir_1_1_ids = []
ir_1_2_ids = []
ir_1_3_ids = []
ir_1_4_ids = []
ir_1_5_ids = []
ir_1_6_ids = []

manual_layers = {

    "ir_2_04B_1": ir_2_04B_1_ids,
    "ir_2_04B_2": ir_2_04B_2_ids,
    "ir_2_04B_3": ir_2_04B_3_ids,
    "ir_2_04B_4": ir_2_04B_4_ids,
    "ir_2_04B_5": ir_2_04B_5_ids,
    "ir_2_04B_6": ir_2_04B_6_ids,
    "ir_1_1": ir_1_1_ids,
    "ir_1_2": ir_1_2_ids,
    "ir_1_3": ir_1_3_ids,
    "ir_1_4": ir_1_4_ids,
    "ir_1_5": ir_1_5_ids,
    "ir_1_6":ir_1_6_ids,

   
}

for name, ids in manual_layers.items():
    layer_name = f"MANC_v1.2.3_{name}"
    color = next(color_cycle)  # or set manually if you want control

    add_segmentation_layer(new_state, layer_name, ids, seg_source, color)



#Add asingle layer manually
# add_segmentation_layer(
#     new_state,
#     "MANC_v1.2.3_candidate_pair",
#     [123456789, 987654321],
#     seg_source,
#     "#FF0000"
# )



# --- Debug print ---
print("State title:", new_state["title"])
print("Number of layers:", len(new_state["layers"]))

for layer in new_state["layers"]:
    print(layer["name"], len(layer.get("segments", [])))


# --- Save ---
encoded = urllib.parse.quote(json.dumps(new_state, separators=(",", ":")))
NEW_URL = "https://clio-ng.janelia.org/#!" + encoded


STATE_OUT = OUTPUT_DIR / "MANC_v1.2.3_4B_04BT1r.json"
URL_OUT = OUTPUT_DIR / "MANC_v1.2.3_4B_04BT1r_url.txt"

with open(STATE_OUT, "w", encoding="utf-8") as f:
    json.dump(new_state, f, indent=2)

with open(URL_OUT, "w", encoding="utf-8") as f:
    f.write(NEW_URL)

print("\nSaved new state to:", STATE_OUT.resolve())
print("Saved URL to:", URL_OUT.resolve())
print("\nNew URL:\n")
print(NEW_URL)

EM source: {'url': 'precomputed://gs://flyem-vnc-2-26-213dba213ef26e094c16c860ae7f4be0/v3_emdata_clahe_xy/jpeg', 'subsources': {'default': True}, 'enableDefaultSubsources': False}
Seg source: [{'url': 'precomputed://gs://manc-seg-v1p2/manc-seg-v1.2', 'subsources': {'default': True, 'mesh': True}, 'enableDefaultSubsources': False}, {'url': 'precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/', 'subsources': {'default': True}, 'enableDefaultSubsources': False}, {'url': 'precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/', 'subsources': {'default': True}, 'enableDefaultSubsources': False}, {'url': 'precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/', 'enableDefaultSubsources': False}]
subclass
IR    59
BR    28
BI     6
IA     4
BA     1
Name: bodyId, dtype: int64
State title: MANC_v1.2.3_4B_T1r
Number of layers: 18
EM 0
MANC_v1.2.3_BA 1
MANC_v1.2.3_BI 6
MANC_v1.2.3_BR 28
MANC_v1.2.3_IA 4
MANC_v1.2.3_

In [8]:
# split out the T3L neurons:

subclass_df = df_4b_t1_rhs.copy()

#retrieve bodyIDs per subclass:
subclass_df["subclass"] = (
    subclass_df["subclass"]
    .fillna("unclassified")
    .astype(str)
    .str.strip()
)
subclass_df.loc[subclass_df["subclass"] == "", "subclass"] = "unclassified"

subclass_to_ids = (
    subclass_df.groupby("subclass")["bodyId"]
    .apply(lambda s: sorted(set(s.dropna().astype(int))))
    .to_dict()
)

##inspect them
for subclass, ids in sorted(subclass_to_ids.items()):
    print(f"{subclass}: {len(ids)} cells")
    print(ids)
    print()

# build layers
for subclass_name, body_ids in sorted(subclass_to_ids.items()):
    layer_name = f"MANC_v1.2.3_{subclass_name}"
    color = next(color_cycle)
    add_segmentation_layer(new_state, layer_name, body_ids, seg_source, color)

BA: 1 cells
[20275]

BI: 6 cells
[11081, 14755, 16504, 16964, 17195, 17407]

BR: 28 cells
[12132, 12949, 13154, 13968, 14137, 14148, 14880, 15750, 16179, 16230, 16626, 17068, 17128, 18269, 18298, 18586, 18608, 19022, 19446, 19614, 20124, 20791, 21212, 21731, 22379, 22954, 38325, 155591]

IA: 4 cells
[10413, 11885, 13414, 19114]

IR: 59 cells
[13465, 14088, 14098, 14297, 14434, 14467, 14811, 14886, 14944, 15369, 15524, 16209, 16938, 17314, 17429, 17584, 17740, 18712, 18796, 18802, 18861, 19167, 19475, 20137, 20520, 20885, 21001, 21134, 21160, 21519, 21782, 22762, 22783, 22787, 22806, 23029, 23152, 23171, 23506, 23545, 23759, 24408, 25160, 25842, 26241, 26261, 26949, 28560, 33190, 33615, 100579, 101895, 102322, 102697, 103220, 153835, 164271, 164272, 166416]



In [9]:
#lets look at IR subclass liek I did for T1- is there dags inthe MANC description? 
#let me inspect subclass IR in 04B to see if MANC has subdivided it 

#fidn the annotations the way I see them when I select a neuron inthe API

from neuprint import fetch_neurons, NeuronCriteria as NC

criteria = NC(bodyId=df_4b_t1_rhs["bodyId"].tolist())

neurons, roi_counts = fetch_neurons(criteria, client=c)

print(neurons.columns)


#------------------------------------
## make neurons_clean should  to handle missing serialMotif values before grouping.

neurons_clean = neurons.copy()

# Flag whether a serial motif was originally annotated
neurons_clean["has_serialMotif"] = neurons_clean["serialMotif"].notna()

# Fill missing values so groupby does not drop them or leave NaN labels
neurons_clean["serialMotif"] = (
    neurons_clean["serialMotif"]
    .fillna("none")
    .astype(str)
    .str.strip()
)

# Optional cleanup for other grouping columns
# for col, default in {
#     "subclass": "unclassified",
#     "type": "unknown",
#     "instance": "none",
# }.items():
#     neurons_clean[col] = neurons_clean[col].fillna(default).astype(str).str.strip()
# table_with_ids = (
#     neurons_clean
#     .groupby(["subclass", "type", "instance", "serialMotif"])
#     .agg(
#         count=("bodyId", "nunique"),
#         bodyIds=("bodyId", lambda x: sorted(set(x)))
#     )
#     .reset_index()
# )
table = (
    neurons_clean
    .groupby(["subclass", "type", "instance", "serialMotif"])["bodyId"]
    .nunique()
    .reset_index(name="count")
)

# add flag AFTER grouping
table["has_motif"] = table["serialMotif"] != "none"

# sort
table = table.sort_values(
    by=["has_motif", "subclass", "count"],
    ascending=[False, True, False]
)

print(table)

#---------------------------------------

ir_df = neurons[neurons["subclass"] == "IR"].copy()
##temporary filtered view

ir_df_selectedcols = ir_df[[
    "bodyId",
    "type",
    "instance",
    "serialMotif",
    "description",
    "modality"
]].sort_values("type")

ir_df_selectedcols

Index(['bodyId', 'instance', 'type', 'pre', 'post', 'downstream', 'upstream',
       'size', 'status', 'statusLabel', 'somaLocation', 'roiInfo', 'prefix',
       'celltypePredictedNt', 'origin', 'somaSide', 'source', 'rootLocation',
       'rootSide', 'modality', 'locationType', 'entryNerve', 'systematicType',
       'predictedNtProb', 'target', 'somaNeuromere', 'tosomaLocation',
       'totalNtPredictions', 'tag', 'predictedNt', 'serial', 'description',
       'receptorType', 'avgLocation', 'synweight', 'subclass',
       'ntAcetylcholineProb', 'subcluster', 'serialMotif', 'ntGlutamateProb',
       'ntGabaProb', 'subclassabbr', 'ntUnknownProb',
       'celltypeTotalNtPredictions', 'group', 'transmission', 'cluster',
       'hemilineage', 'longTract', 'synonyms', 'class', 'location',
       'exitNerve', 'birthtime', 'vfbId', 'inputRois', 'outputRois'],
      dtype='object')
   subclass      type       instance      serialMotif  count  has_motif
5        BR  IN04B008  IN04B008_T1_R  ind

,bodyId,type,instance,serialMotif,description,modality
9,14088,IN04B009,IN04B009_T1_R,None,20A in T1 RHS,None
14,14434,IN04B009,IN04B009_T1_R,None,20A in T1 RHS,None
21,15369,IN04B009,IN04B009_T1_R,None,20A in T1 RHS,None
92,103220,IN04B014,IN04B014_T1_R,None,1B?? in T1 RHS,None
17,14811,IN04B015,IN04B015_T1_R,None,None,None
22,15524,IN04B015,IN04B015_T1_R,None,None,None
25,16209,IN04B015,IN04B015_T1_R,None,None,None
15,14467,IN04B025,IN04B025_T1_R,independent leg,20A in T1 RHS,None
7,13465,IN04B026,IN04B026_T1_R,independent leg,20A? in T1 RHS,None
81,26241,IN04B031,IN04B031_T1_R,independent leg,4B? in T1 RHS,None


In [10]:
#ok let us make two Dfs for the 04bIR and the 20A flagged 04B IRs in T3L

#let me inspect subclass IR in 04B to see if MANC has subdivided it 

###pull all IR neurons in ir_df
ir_df = neurons[neurons["subclass"] == "IR"].copy()


ir_df[[
    "bodyId",
    "type",
    "instance",
    "serialMotif",
    "description",
    "modality"
]].sort_values("type")

ir_ids = sorted(ir_df["bodyId"].dropna().astype(int).unique().tolist())
print(ir_ids)
print("n IR neurons:", len(ir_ids))


# create flag
ir_df["maybe_20A"] = ir_df["description"].str.contains(
    "20A", case=False, na=False
)

# subset
ir_20A = ir_df[ir_df["maybe_20A"]]

ir_20A[[
    "bodyId", "type", "instance", "description", 'serialMotif'
]]

# everything else
ir_04B = ir_df[~ir_df["maybe_20A"]]



# make unique type lists for the two IR groups

types_20A = (
    ir_20A
    .groupby("type")["bodyId"]
    .agg(lambda x: sorted(set(x)))
    .reset_index(name="bodyIds")
    .sort_values("type")
    .reset_index(drop=True)
)

types_04B_or_unknown = (
    ir_04B
    .groupby("type")["bodyId"]
    .agg(lambda x: sorted(set(x)))
    .reset_index(name="bodyIds")
    .sort_values("type")
    .reset_index(drop=True)
)

types_20A
types_04B_or_unknown

types_20A_set = set(types_20A["type"])
types_04B_set = set(types_04B_or_unknown["type"])

overlap = types_20A_set & types_04B_set
only_20A = types_20A_set - types_04B_set
only_04B = types_04B_set - types_20A_set

print("Overlap:", overlap)
print("\nOnly in 20A:", only_20A)
print("\nOnly in 04B/unknown:", only_04B)


n_20A = ir_df[ir_df["maybe_20A"]]["bodyId"].nunique()
n_04B = ir_df[~ir_df["maybe_20A"]]["bodyId"].nunique()

print("IR maybe 20A:", n_20A)
print("IR 04B/unknown:", n_04B)

ids_20A = [int(x) for x in ir_20A["bodyId"].unique()]
ids_04B = [int(x) for x in ir_04B["bodyId"].unique()]

print(ids_20A)
print(ids_04B)



[13465, 14088, 14098, 14297, 14434, 14467, 14811, 14886, 14944, 15369, 15524, 16209, 16938, 17314, 17429, 17584, 17740, 18712, 18796, 18802, 18861, 19167, 19475, 20137, 20520, 20885, 21001, 21134, 21160, 21519, 21782, 22762, 22783, 22787, 22806, 23029, 23152, 23171, 23506, 23545, 23759, 24408, 25160, 25842, 26241, 26261, 26949, 28560, 33190, 33615, 100579, 101895, 102322, 102697, 103220, 153835, 164271, 164272, 166416]
n IR neurons: 59
Overlap: {'IN04B059', 'IN04B081'}

Only in 20A: {'IN04B102', 'IN04B092', 'IN04B101', 'IN04B098', 'IN04B095', 'IN04B009', 'IN04B104', 'IN04B093', 'IN04B070', 'IN04B097', 'IN04B115', 'IN04B026', 'IN04B112', 'IN04B025'}

Only in 04B/unknown: {'IN04B015', 'IN04B037', 'IN04B034', 'IN04B111', 'IN04B072', 'IN04B078', 'IN04B066', 'IN04B053', 'IN04B091', 'IN04B094', 'IN04B014', 'IN04B038', 'IN04B085', 'IN04B069', 'IN04B100', 'IN04B031'}
IR maybe 20A: 26
IR 04B/unknown: 33
[13465, 14088, 14434, 14467, 14944, 15369, 17314, 17584, 17740, 20137, 20885, 21001, 21134, 

In [12]:
#Jelly manual cluster annotation- reviewed in neuroglancer for morphologies
ir_all = [13465, 14088, 14098, 14297, 14434, 14467, 14811, 14886, 14944, 15369, 15524, 16209, 16938, 17314, 17429, 17584, 17740, 18712, 18796, 18802, 18861, 19167, 19475, 20137, 20520, 20885, 21001, 21134, 21160, 21519, 21782, 22762, 22783, 22787, 22806, 23029, 23152, 23171, 23506, 23545, 23759, 24408, 25160, 25842, 26241, 26261, 26949, 28560, 33190, 33615, 100579, 101895, 102322, 102697, 103220, 153835, 164271, 164272, 166416]



ir_20A = [13465, 14088, 14434, 14467, 14944, 15369, 17314, 17584, 17740, 20137, 20885, 21001, 21134, 21782, 22762, 22783, 22787, 23029, 23152, 23171, 23545, 24408, 25160, 28560, 33190, 166416]



print (len(ir_all))

ir_04B = [14098, 14297, 14811, 14886, 15524, 16209, 16938, 17429, 18712, 18796, 18802, 18861, 19167, 19475, 20520, 21160, 21519, 22806, 23506, 23759, 25842, 26241, 26261, 26949, 33615, 100579, 101895, 102322, 102697, 103220, 153835, 164271, 164272]


print (len(ir_04B))

#Jelly clusters: 
ir_2_04B_1_ids = [14811, 15524, 16209, 21519]
ir_2_04B_2_ids = [14098, 19475, 22806, 23506, 26261, 26949, 164272]
ir_2_04B_3_ids = [19167, 33615, 164271]
ir_2_04B_4_ids = [14297, 153835]
ir_2_04B_5_ids = [14886, 18712, 20520, 23759, 25842, 26241, 100579, 101895, 102697]
ir_2_04B_6_ids = [16938, 18796, 18802, 21160]
ir_2_04B_7_ids = [17429, 18861, 102322]
ir_2_04B_8_ids = [103220]
ir_remaining = [x for x in ir_04B if x not in ir_2_04B_1_ids and x not in ir_2_04B_2_ids and x not in ir_2_04B_3_ids and x not in ir_2_04B_4_ids and x not in ir_2_04B_5_ids and x not in ir_2_04B_6_ids and x not in ir_2_04B_7_ids]
#ir_remaining = [x for x in ir_04B if x not in ir_2_04B_1_ids and x not in ir_2_04B_2_ids and x not in ir_2_04B_4_ids and x not in ir_2_04B_5_ids]

print(ir_remaining)



# # #did I miss anything: 
all_groups = (
    set(ir_2_04B_1_ids)
    | set(ir_2_04B_2_ids)
    | set(ir_2_04B_3_ids)
    | set(ir_2_04B_4_ids)
    | set(ir_2_04B_5_ids)
    | set(ir_2_04B_6_ids)
    | set(ir_2_04B_7_ids)
)


ir_set = set(ir_04B)

missing = ir_set - all_groups
extra = all_groups - ir_set

print("Missing from groups:", missing)
print("Extra in groups:", extra)


#did I duplicate any neuron?
from collections import Counter

all_ids_list = (
    ir_2_04B_1_ids
    + ir_2_04B_2_ids
    + ir_2_04B_3_ids
    + ir_2_04B_4_ids
    + ir_2_04B_5_ids
    + ir_2_04B_6_ids
    + ir_2_04B_7_ids
)

counts = Counter(all_ids_list)

duplicates = {k: v for k, v in counts.items() if v > 1}
print("Duplicates across groups:", duplicates)



59
33
[103220]
Missing from groups: {103220}
Extra in groups: set()
Duplicates across groups: {}


In [22]:
#beautiful. Now we do group T1l-IR tagged as 20B

#Jelly manual cluster annotation- reviewed in neuroglancer for morphologies
ir_all = [13465, 14088, 14098, 14297, 14434, 14467, 14811, 14886, 14944, 15369, 15524, 16209, 16938, 17314, 17429, 17584, 17740, 18712, 18796, 18802, 18861, 19167, 19475, 20137, 20520, 20885, 21001, 21134, 21160, 21519, 21782, 22762, 22783, 22787, 22806, 23029, 23152, 23171, 23506, 23545, 23759, 24408, 25160, 25842, 26241, 26261, 26949, 28560, 33190, 33615, 100579, 101895, 102322, 102697, 103220, 153835, 164271, 164272, 166416]
ir_20A = [13465, 14088, 14434, 14467, 14944, 15369, 17314, 17584, 17740, 20137, 20885, 21001, 21134, 21782, 22762, 22783, 22787, 23029, 23152, 23171, 23545, 24408, 25160, 28560, 33190, 166416]
print (len(ir_all))
ir_04B = [14098, 14297, 14811, 14886, 15524, 16209, 16938, 17429, 18712, 18796, 18802, 18861, 19167, 19475, 20520, 21160, 21519, 22806, 23506, 23759, 25842, 26241, 26261, 26949, 33615, 100579, 101895, 102322, 102697, 103220, 153835, 164271, 164272]
print (len(ir_04B))

#ir_20A = [x for x in ir_all if x not in ir_04B]

print(len(ir_20A))
print("IDs in 20A")
print (ir_20A)

ir_1_1_ids = [13465, 14088, 14434, 15369, 14467]

ir_1_2_ids = [17314, 17584, 21134, 21782, 22762, 22783, 23171, 23545, 33190, 166416, 17740]
ir_1_3_ids = [20885, 22787, 23029, 23152, 28560]

ir_1_4_ids = [21001, 25160]
ir_1_5_ids = [14944, 20137, 24408]
ir_1_6_ids = []
ir_1_remaining = [x for x in ir_20A if x not in ir_1_1_ids and x not in ir_1_2_ids and x not in ir_1_3_ids and x not in ir_1_4_ids and x not in ir_1_5_ids]

#ir_1_remaining = [x for x in ir_20A if x not in ir_1_1_ids and x not in ir_1_2_ids and x not in ir_1_3_ids and x not in ir_1_4_ids and x not in ir_1_5_ids] 
print("IDs IR remaining:", ir_1_remaining)

# #did I miss anything: 
all_groups = (
    set(ir_1_1_ids)
    | set(ir_1_2_ids)
    | set(ir_1_3_ids)
    | set(ir_1_4_ids)
    | set(ir_1_5_ids)
    | set(ir_1_6_ids)
  
)


ir_set_20A = set(ir_20A)

missing = ir_set_20A - all_groups
extra = all_groups - ir_set_20A

print("Missing from groups:", missing)
print("Extra in groups:", extra)


#did I duplicate any neuron?
from collections import Counter

all_ids_list = (
    ir_1_1_ids
    + ir_1_2_ids
    + ir_1_3_ids
    + ir_1_4_ids
    + ir_1_5_ids
    + ir_1_6_ids
  
)

counts = Counter(all_ids_list)

duplicates = {k: v for k, v in counts.items() if v > 1}
print("Duplicates across groups:", duplicates)




59
33
26
IDs in 20A
[13465, 14088, 14434, 14467, 14944, 15369, 17314, 17584, 17740, 20137, 20885, 21001, 21134, 21782, 22762, 22783, 22787, 23029, 23152, 23171, 23545, 24408, 25160, 28560, 33190, 166416]
IDs IR remaining: []
Missing from groups: set()
Extra in groups: set()
Duplicates across groups: {}


#final sort
https://clio-ng.janelia.org/#!%7B%22title%22:%22MANC_v1.2.3_4B_T1r%22%2C%22dimensions%22:%7B%22x%22:%5B8e-9%2C%22m%22%5D%2C%22y%22:%5B8e-9%2C%22m%22%5D%2C%22z%22:%5B8e-9%2C%22m%22%5D%7D%2C%22position%22:%5B27407.5%2C29682.5%2C50420.5%5D%2C%22crossSectionOrientation%22:%5B0%2C0.7071067690849304%2C-0.7071067690849304%2C0%5D%2C%22crossSectionScale%22:1%2C%22projectionOrientation%22:%5B-0.3532412350177765%2C0.5382871627807617%2C-0.6803406476974487%2C0.3501487672328949%5D%2C%22projectionScale%22:27518.321388470133%2C%22layers%22:%5B%7B%22type%22:%22image%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-2-26-213dba213ef26e094c16c860ae7f4be0/v3_emdata_clahe_xy/jpeg%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22tab%22:%22source%22%2C%22name%22:%22EM%22%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2220275%22%5D%2C%22segmentQuery%22:%22https://clio-ng.janelia.org/#%21%257B%2522title%2522%253A%2522MANC_v1.2.3_4B_T1r%2522%252C%2522dimensions%2522%253A%257B%2522x%2522%253A%255B8e-09%252C%2522m%2522%255D%252C%2522y%2522%253A%255B8e-09%252C%2522m%2522%255D%252C%2522z%2522%253A%255B8e-09%252C%2522m%2522%255D%257D%252C%2522position%2522%253A%255B23056.5%252C29733.5%252C41138.5%255D%252C%2522crossSectionOrientation%2522%253A%255B0%252C0.7071067690849304%252C-0.7071067690849304%252C0%255D%252C%2522crossSectionScale%2522%253A1%252C%2522projectionOrientation%2522%253A%255B0%252C0.7071067690849304%252C-0.7071067690849304%252C0%255D%252C%2522projectionScale%2522%253A91364.04452716278%252C%2522layout%2522%253A%25223d%2522%252C%2522showSlices%2522%253Afalse%252C%2522projectionBackgroundColor%2522%253A%2522%2523ffffff%2522%252C%2522crossSectionBackgroundColor%2522%253A%2522%2523ffffff%2522%252C%2522selectedLayer%2522%253A%257B%2522flex%2522%253A1.49%252C%2522size%2522%253A426%252C%2522visible%2522%253Atrue%252C%2522layer%2522%253A%2522manc%253Av1.2.3%2522%257D%252C%2522settingsPanel%2522%253A%257B%2522visible%2522%253Atrue%257D%252C%2522layers%2522%253A%255B%257B%2522type%2522%253A%2522image%2522%252C%2522name%2522%253A%2522EM%2522%252C%2522source%2522%253A%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//flyem-vnc-2-26-213dba213ef26e094c16c860ae7f4be0/v3_emdata_clahe_xy/jpeg%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%2522visible%2522%253Atrue%257D%252C%257B%2522type%2522%253A%2522segmentation%2522%252C%2522name%2522%253A%2522MANC_v1.2.3_BA%2522%252C%2522source%2522%253A%255B%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%252C%2522mesh%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%2522%252C%2522enableDefaultSubsources%2522%253Afalse%257D%255D%252C%2522segments%2522%253A%255B%252220275%2522%255D%252C%2522segmentColors%2522%253A%257B%252220275%2522%253A%2522%252387CEEB%2522%257D%252C%2522visible%2522%253Atrue%257D%252C%257B%2522type%2522%253A%2522segmentation%2522%252C%2522name%2522%253A%2522MANC_v1.2.3_BI%2522%252C%2522source%2522%253A%255B%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%252C%2522mesh%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%2522%252C%2522enableDefaultSubsources%2522%253Afalse%257D%255D%252C%2522segments%2522%253A%255B%252211081%2522%252C%252214755%2522%252C%252216504%2522%252C%252216964%2522%252C%252217195%2522%252C%252217407%2522%255D%252C%2522segmentColors%2522%253A%257B%252211081%2522%253A%2522%2523FFA500%2522%252C%252214755%2522%253A%2522%2523FFA500%2522%252C%252216504%2522%253A%2522%2523FFA500%2522%252C%252216964%2522%253A%2522%2523FFA500%2522%252C%252217195%2522%253A%2522%2523FFA500%2522%252C%252217407%2522%253A%2522%2523FFA500%2522%257D%252C%2522visible%2522%253Atrue%257D%252C%257B%2522type%2522%253A%2522segmentation%2522%252C%2522name%2522%253A%2522MANC_v1.2.3_BR%2522%252C%2522source%2522%253A%255B%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%252C%2522mesh%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%2522%252C%2522enableDefaultSubsources%2522%253Afalse%257D%255D%252C%2522segments%2522%253A%255B%252212132%2522%252C%252212949%2522%252C%252213154%2522%252C%252213968%2522%252C%252214137%2522%252C%252214148%2522%252C%252214880%2522%252C%252215750%2522%252C%252216179%2522%252C%252216230%2522%252C%252216626%2522%252C%252217068%2522%252C%252217128%2522%252C%252218269%2522%252C%252218298%2522%252C%252218586%2522%252C%252218608%2522%252C%252219022%2522%252C%252219446%2522%252C%252219614%2522%252C%252220124%2522%252C%252220791%2522%252C%252221212%2522%252C%252221731%2522%252C%252222379%2522%252C%252222954%2522%252C%252238325%2522%252C%2522155591%2522%255D%252C%2522segmentColors%2522%253A%257B%252212132%2522%253A%2522%2523E34234%2522%252C%252212949%2522%253A%2522%2523E34234%2522%252C%252213154%2522%253A%2522%2523E34234%2522%252C%252213968%2522%253A%2522%2523E34234%2522%252C%252214137%2522%253A%2522%2523E34234%2522%252C%252214148%2522%253A%2522%2523E34234%2522%252C%252214880%2522%253A%2522%2523E34234%2522%252C%252215750%2522%253A%2522%2523E34234%2522%252C%252216179%2522%253A%2522%2523E34234%2522%252C%252216230%2522%253A%2522%2523E34234%2522%252C%252216626%2522%253A%2522%2523E34234%2522%252C%252217068%2522%253A%2522%2523E34234%2522%252C%252217128%2522%253A%2522%2523E34234%2522%252C%252218269%2522%253A%2522%2523E34234%2522%252C%252218298%2522%253A%2522%2523E34234%2522%252C%252218586%2522%253A%2522%2523E34234%2522%252C%252218608%2522%253A%2522%2523E34234%2522%252C%252219022%2522%253A%2522%2523E34234%2522%252C%252219446%2522%253A%2522%2523E34234%2522%252C%252219614%2522%253A%2522%2523E34234%2522%252C%252220124%2522%253A%2522%2523E34234%2522%252C%252220791%2522%253A%2522%2523E34234%2522%252C%252221212%2522%253A%2522%2523E34234%2522%252C%252221731%2522%253A%2522%2523E34234%2522%252C%252222379%2522%253A%2522%2523E34234%2522%252C%252222954%2522%253A%2522%2523E34234%2522%252C%252238325%2522%253A%2522%2523E34234%2522%252C%2522155591%2522%253A%2522%2523E34234%2522%257D%252C%2522visible%2522%253Atrue%257D%252C%257B%2522type%2522%253A%2522segmentation%2522%252C%2522name%2522%253A%2522MANC_v1.2.3_IA%2522%252C%2522source%2522%253A%255B%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%252C%2522mesh%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%2522%252C%2522enableDefaultSubsources%2522%253Afalse%257D%255D%252C%2522segments%2522%253A%255B%252210413%2522%252C%252211885%2522%252C%252213414%2522%252C%252219114%2522%255D%252C%2522segmentColors%2522%253A%257B%252210413%2522%253A%2522%2523CC99FF%2522%252C%252211885%2522%253A%2522%2523CC99FF%2522%252C%252213414%2522%253A%2522%2523CC99FF%2522%252C%252219114%2522%253A%2522%2523CC99FF%2522%257D%252C%2522visible%2522%253Atrue%257D%252C%257B%2522type%2522%253A%2522segmentation%2522%252C%2522name%2522%253A%2522MANC_v1.2.3_IR%2522%252C%2522source%2522%253A%255B%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%252C%2522mesh%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%2522%252C%2522enableDefaultSubsources%2522%253Afalse%257D%255D%252C%2522segments%2522%253A%255B%252213465%2522%252C%252214088%2522%252C%252214098%2522%252C%252214297%2522%252C%252214434%2522%252C%252214467%2522%252C%252214811%2522%252C%252214886%2522%252C%252214944%2522%252C%252215369%2522%252C%252215524%2522%252C%252216209%2522%252C%252216938%2522%252C%252217314%2522%252C%252217429%2522%252C%252217584%2522%252C%252217740%2522%252C%252218712%2522%252C%252218796%2522%252C%252218802%2522%252C%252218861%2522%252C%252219167%2522%252C%252219475%2522%252C%252220137%2522%252C%252220520%2522%252C%252220885%2522%252C%252221001%2522%252C%252221134%2522%252C%252221160%2522%252C%252221519%2522%252C%252221782%2522%252C%252222762%2522%252C%252222783%2522%252C%252222787%2522%252C%252222806%2522%252C%252223029%2522%252C%252223152%2522%252C%252223171%2522%252C%252223506%2522%252C%252223545%2522%252C%252223759%2522%252C%252224408%2522%252C%252225160%2522%252C%252225842%2522%252C%252226241%2522%252C%252226261%2522%252C%252226949%2522%252C%252228560%2522%252C%252233190%2522%252C%252233615%2522%252C%2522100579%2522%252C%2522101895%2522%252C%2522102322%2522%252C%2522102697%2522%252C%2522103220%2522%252C%2522153835%2522%252C%2522164271%2522%252C%2522164272%2522%252C%2522166416%2522%255D%252C%2522segmentColors%2522%253A%257B%252213465%2522%253A%2522%252366CDAA%2522%252C%252214088%2522%253A%2522%252366CDAA%2522%252C%252214098%2522%253A%2522%252366CDAA%2522%252C%252214297%2522%253A%2522%252366CDAA%2522%252C%252214434%2522%253A%2522%252366CDAA%2522%252C%252214467%2522%253A%2522%252366CDAA%2522%252C%252214811%2522%253A%2522%252366CDAA%2522%252C%252214886%2522%253A%2522%252366CDAA%2522%252C%252214944%2522%253A%2522%252366CDAA%2522%252C%252215369%2522%253A%2522%252366CDAA%2522%252C%252215524%2522%253A%2522%252366CDAA%2522%252C%252216209%2522%253A%2522%252366CDAA%2522%252C%252216938%2522%253A%2522%252366CDAA%2522%252C%252217314%2522%253A%2522%252366CDAA%2522%252C%252217429%2522%253A%2522%252366CDAA%2522%252C%252217584%2522%253A%2522%252366CDAA%2522%252C%252217740%2522%253A%2522%252366CDAA%2522%252C%252218712%2522%253A%2522%252366CDAA%2522%252C%252218796%2522%253A%2522%252366CDAA%2522%252C%252218802%2522%253A%2522%252366CDAA%2522%252C%252218861%2522%253A%2522%252366CDAA%2522%252C%252219167%2522%253A%2522%252366CDAA%2522%252C%252219475%2522%253A%2522%252366CDAA%2522%252C%252220137%2522%253A%2522%252366CDAA%2522%252C%252220520%2522%253A%2522%252366CDAA%2522%252C%252220885%2522%253A%2522%252366CDAA%2522%252C%252221001%2522%253A%2522%252366CDAA%2522%252C%252221134%2522%253A%2522%252366CDAA%2522%252C%252221160%2522%253A%2522%252366CDAA%2522%252C%252221519%2522%253A%2522%252366CDAA%2522%252C%252221782%2522%253A%2522%252366CDAA%2522%252C%252222762%2522%253A%2522%252366CDAA%2522%252C%252222783%2522%253A%2522%252366CDAA%2522%252C%252222787%2522%253A%2522%252366CDAA%2522%252C%252222806%2522%253A%2522%252366CDAA%2522%252C%252223029%2522%253A%2522%252366CDAA%2522%252C%252223152%2522%253A%2522%252366CDAA%2522%252C%252223171%2522%253A%2522%252366CDAA%2522%252C%252223506%2522%253A%2522%252366CDAA%2522%252C%252223545%2522%253A%2522%252366CDAA%2522%252C%252223759%2522%253A%2522%252366CDAA%2522%252C%252224408%2522%253A%2522%252366CDAA%2522%252C%252225160%2522%253A%2522%252366CDAA%2522%252C%252225842%2522%253A%2522%252366CDAA%2522%252C%252226241%2522%253A%2522%252366CDAA%2522%252C%252226261%2522%253A%2522%252366CDAA%2522%252C%252226949%2522%253A%2522%252366CDAA%2522%252C%252228560%2522%253A%2522%252366CDAA%2522%252C%252233190%2522%253A%2522%252366CDAA%2522%252C%252233615%2522%253A%2522%252366CDAA%2522%252C%2522100579%2522%253A%2522%252366CDAA%2522%252C%2522101895%2522%253A%2522%252366CDAA%2522%252C%2522102322%2522%253A%2522%252366CDAA%2522%252C%2522102697%2522%253A%2522%252366CDAA%2522%252C%2522103220%2522%253A%2522%252366CDAA%2522%252C%2522153835%2522%253A%2522%252366CDAA%2522%252C%2522164271%2522%253A%2522%252366CDAA%2522%252C%2522164272%2522%253A%2522%252366CDAA%2522%252C%2522166416%2522%253A%2522%252366CDAA%2522%257D%252C%2522visible%2522%253Atrue%257D%252C%257B%2522type%2522%253A%2522segmentation%2522%252C%2522name%2522%253A%2522MANC_v1.2.3_ir_2_04B_1%2522%252C%2522source%2522%253A%255B%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%252C%2522mesh%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%2522%252C%2522enableDefaultSubsources%2522%253Afalse%257D%255D%252C%2522segments%2522%253A%255B%255D%252C%2522segmentColors%2522%253A%257B%257D%252C%2522visible%2522%253Atrue%257D%252C%257B%2522type%2522%253A%2522segmentation%2522%252C%2522name%2522%253A%2522MANC_v1.2.3_ir_2_04B_2%2522%252C%2522source%2522%253A%255B%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%252C%2522mesh%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%2522%252C%2522enableDefaultSubsources%2522%253Afalse%257D%255D%252C%2522segments%2522%253A%255B%255D%252C%2522segmentColors%2522%253A%257B%257D%252C%2522visible%2522%253Atrue%257D%252C%257B%2522type%2522%253A%2522segmentation%2522%252C%2522name%2522%253A%2522MANC_v1.2.3_ir_2_04B_3%2522%252C%2522source%2522%253A%255B%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%252C%2522mesh%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%2522%252C%2522enableDefaultSubsources%2522%253Afalse%257D%255D%252C%2522segments%2522%253A%255B%255D%252C%2522segmentColors%2522%253A%257B%257D%252C%2522visible%2522%253Atrue%257D%252C%257B%2522type%2522%253A%2522segmentation%2522%252C%2522name%2522%253A%2522MANC_v1.2.3_ir_2_04B_4%2522%252C%2522source%2522%253A%255B%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%252C%2522mesh%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%2522%252C%2522enableDefaultSubsources%2522%253Afalse%257D%255D%252C%2522segments%2522%253A%255B%255D%252C%2522segmentColors%2522%253A%257B%257D%252C%2522visible%2522%253Atrue%257D%252C%257B%2522type%2522%253A%2522segmentation%2522%252C%2522name%2522%253A%2522MANC_v1.2.3_ir_2_04B_5%2522%252C%2522source%2522%253A%255B%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%252C%2522mesh%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%2522%252C%2522enableDefaultSubsources%2522%253Afalse%257D%255D%252C%2522segments%2522%253A%255B%255D%252C%2522segmentColors%2522%253A%257B%257D%252C%2522visible%2522%253Atrue%257D%252C%257B%2522type%2522%253A%2522segmentation%2522%252C%2522name%2522%253A%2522MANC_v1.2.3_ir_2_04B_6%2522%252C%2522source%2522%253A%255B%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%252C%2522mesh%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%2522%252C%2522enableDefaultSubsources%2522%253Afalse%257D%255D%252C%2522segments%2522%253A%255B%255D%252C%2522segmentColors%2522%253A%257B%257D%252C%2522visible%2522%253Atrue%257D%252C%257B%2522type%2522%253A%2522segmentation%2522%252C%2522name%2522%253A%2522MANC_v1.2.3_ir_1_1%2522%252C%2522source%2522%253A%255B%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%252C%2522mesh%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%2522%252C%2522enableDefaultSubsources%2522%253Afalse%257D%255D%252C%2522segments%2522%253A%255B%255D%252C%2522segmentColors%2522%253A%257B%257D%252C%2522visible%2522%253Atrue%257D%252C%257B%2522type%2522%253A%2522segmentation%2522%252C%2522name%2522%253A%2522MANC_v1.2.3_ir_1_2%2522%252C%2522source%2522%253A%255B%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%252C%2522mesh%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%2522%252C%2522enableDefaultSubsources%2522%253Afalse%257D%255D%252C%2522segments%2522%253A%255B%255D%252C%2522segmentColors%2522%253A%257B%257D%252C%2522visible%2522%253Atrue%257D%252C%257B%2522type%2522%253A%2522segmentation%2522%252C%2522name%2522%253A%2522MANC_v1.2.3_ir_1_3%2522%252C%2522source%2522%253A%255B%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%252C%2522mesh%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%2522%252C%2522enableDefaultSubsources%2522%253Afalse%257D%255D%252C%2522segments%2522%253A%255B%255D%252C%2522segmentColors%2522%253A%257B%257D%252C%2522visible%2522%253Atrue%257D%252C%257B%2522type%2522%253A%2522segmentation%2522%252C%2522name%2522%253A%2522MANC_v1.2.3_ir_1_4%2522%252C%2522source%2522%253A%255B%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%252C%2522mesh%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%2522%252C%2522enableDefaultSubsources%2522%253Afalse%257D%255D%252C%2522segments%2522%253A%255B%255D%252C%2522segmentColors%2522%253A%257B%257D%252C%2522visible%2522%253Atrue%257D%252C%257B%2522type%2522%253A%2522segmentation%2522%252C%2522name%2522%253A%2522MANC_v1.2.3_ir_1_5%2522%252C%2522source%2522%253A%255B%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%252C%2522mesh%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%2522%252C%2522enableDefaultSubsources%2522%253Afalse%257D%255D%252C%2522segments%2522%253A%255B%255D%252C%2522segmentColors%2522%253A%257B%257D%252C%2522visible%2522%253Atrue%257D%252C%257B%2522type%2522%253A%2522segmentation%2522%252C%2522name%2522%253A%2522MANC_v1.2.3_ir_1_6%2522%252C%2522source%2522%253A%255B%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%252C%2522mesh%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%2522%252C%2522subsources%2522%253A%257B%2522default%2522%253Atrue%257D%252C%2522enableDefaultSubsources%2522%253Afalse%257D%252C%257B%2522url%2522%253A%2522precomputed%253A//gs%253A//manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%2522%252C%2522enableDefaultSubsources%2522%253Afalse%257D%255D%252C%2522segments%2522%253A%255B%255D%252C%2522segmentColors%2522%253A%257B%257D%252C%2522visible%2522%253Atrue%257D%255D%257D%22%2C%22segmentColors%22:%7B%2220275%22:%22#87ceeb%22%7D%2C%22name%22:%22MANC_v1.2.3_BA%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2211081%22%2C%2214755%22%2C%2216504%22%2C%2216964%22%2C%2217195%22%2C%2217407%22%5D%2C%22segmentQuery%22:%2211081%2C%2014755%2C%2016504%2C%2016964%2C%2017195%2C%2017407%22%2C%22segmentColors%22:%7B%2211081%22:%22#ffa500%22%2C%2214755%22:%22#ffa500%22%2C%2216504%22:%22#ffa500%22%2C%2216964%22:%22#ffa500%22%2C%2217195%22:%22#ffa500%22%2C%2217407%22:%22#ffa500%22%7D%2C%22name%22:%22MANC_v1.2.3_BI%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2212132%22%2C%2212949%22%2C%2213154%22%2C%2213968%22%2C%2214137%22%2C%2214148%22%2C%2214880%22%2C%22155591%22%2C%2215750%22%2C%2216179%22%2C%2216230%22%2C%2216626%22%2C%2217068%22%2C%2217128%22%2C%2218269%22%2C%2218298%22%2C%2218586%22%2C%2218608%22%2C%2219022%22%2C%2219446%22%2C%2219614%22%2C%2220124%22%2C%2220791%22%2C%2221212%22%2C%2221731%22%2C%2222379%22%2C%2222954%22%2C%2238325%22%5D%2C%22segmentQuery%22:%2212132%2C%2012949%2C%2013154%2C%2013968%2C%2014137%2C%2014148%2C%2014880%2C%2015750%2C%2016179%2C%2016230%2C%2016626%2C%2017068%2C%2017128%2C%2018269%2C%2018298%2C%2018586%2C%2018608%2C%2019022%2C%2019446%2C%2019614%2C%2020124%2C%2020791%2C%2021212%2C%2021731%2C%2022379%2C%2022954%2C%2038325%2C%20155591%22%2C%22segmentColors%22:%7B%2212132%22:%22#e34234%22%2C%2212949%22:%22#e34234%22%2C%2213154%22:%22#e34234%22%2C%2213968%22:%22#e34234%22%2C%2214137%22:%22#e34234%22%2C%2214148%22:%22#e34234%22%2C%2214880%22:%22#e34234%22%2C%2215750%22:%22#e34234%22%2C%2216179%22:%22#e34234%22%2C%2216230%22:%22#e34234%22%2C%2216626%22:%22#e34234%22%2C%2217068%22:%22#e34234%22%2C%2217128%22:%22#e34234%22%2C%2218269%22:%22#e34234%22%2C%2218298%22:%22#e34234%22%2C%2218586%22:%22#e34234%22%2C%2218608%22:%22#e34234%22%2C%2219022%22:%22#e34234%22%2C%2219446%22:%22#e34234%22%2C%2219614%22:%22#e34234%22%2C%2220124%22:%22#e34234%22%2C%2220791%22:%22#e34234%22%2C%2221212%22:%22#e34234%22%2C%2221731%22:%22#e34234%22%2C%2222379%22:%22#e34234%22%2C%2222954%22:%22#e34234%22%2C%2238325%22:%22#e34234%22%2C%22155591%22:%22#e34234%22%7D%2C%22name%22:%22MANC_v1.2.3_BR%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2210413%22%2C%2211885%22%2C%2213414%22%2C%2219114%22%5D%2C%22segmentQuery%22:%2210413%2C%2011885%2C%2013414%2C%2019114%22%2C%22segmentColors%22:%7B%2210413%22:%22#cc99ff%22%2C%2211885%22:%22#cc99ff%22%2C%2213414%22:%22#cc99ff%22%2C%2219114%22:%22#cc99ff%22%7D%2C%22name%22:%22MANC_v1.2.3_IA%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%22100579%22%2C%22101895%22%2C%22102322%22%2C%22102697%22%2C%22103220%22%2C%2213465%22%2C%2214088%22%2C%2214098%22%2C%2214297%22%2C%2214434%22%2C%2214467%22%2C%2214811%22%2C%2214886%22%2C%2214944%22%2C%2215369%22%2C%22153835%22%2C%2215524%22%2C%2216209%22%2C%22164271%22%2C%22164272%22%2C%22166416%22%2C%2216938%22%2C%2217314%22%2C%2217429%22%2C%2217584%22%2C%2217740%22%2C%2218712%22%2C%2218796%22%2C%2218802%22%2C%2218861%22%2C%2219167%22%2C%2219475%22%2C%2220137%22%2C%2220520%22%2C%2220885%22%2C%2221001%22%2C%2221134%22%2C%2221160%22%2C%2221519%22%2C%2221782%22%2C%2222762%22%2C%2222783%22%2C%2222787%22%2C%2222806%22%2C%2223029%22%2C%2223152%22%2C%2223171%22%2C%2223506%22%2C%2223545%22%2C%2223759%22%2C%2224408%22%2C%2225160%22%2C%2225842%22%2C%2226241%22%2C%2226261%22%2C%2226949%22%2C%2228560%22%2C%2233190%22%2C%2233615%22%5D%2C%22segmentQuery%22:%2213465%2C%2014088%2C%2014098%2C%2014297%2C%2014434%2C%2014467%2C%2014811%2C%2014886%2C%2014944%2C%2015369%2C%2015524%2C%2016209%2C%2016938%2C%2017314%2C%2017429%2C%2017584%2C%2017740%2C%2018712%2C%2018796%2C%2018802%2C%2018861%2C%2019167%2C%2019475%2C%2020137%2C%2020520%2C%2020885%2C%2021001%2C%2021134%2C%2021160%2C%2021519%2C%2021782%2C%2022762%2C%2022783%2C%2022787%2C%2022806%2C%2023029%2C%2023152%2C%2023171%2C%2023506%2C%2023545%2C%2023759%2C%2024408%2C%2025160%2C%2025842%2C%2026241%2C%2026261%2C%2026949%2C%2028560%2C%2033190%2C%2033615%2C%20100579%2C%20101895%2C%20102322%2C%20102697%2C%20103220%2C%20153835%2C%20164271%2C%20164272%2C%20166416%22%2C%22segmentColors%22:%7B%2213465%22:%22#66cdaa%22%2C%2214088%22:%22#66cdaa%22%2C%2214098%22:%22#66cdaa%22%2C%2214297%22:%22#66cdaa%22%2C%2214434%22:%22#66cdaa%22%2C%2214467%22:%22#66cdaa%22%2C%2214811%22:%22#66cdaa%22%2C%2214886%22:%22#66cdaa%22%2C%2214944%22:%22#66cdaa%22%2C%2215369%22:%22#66cdaa%22%2C%2215524%22:%22#66cdaa%22%2C%2216209%22:%22#66cdaa%22%2C%2216938%22:%22#66cdaa%22%2C%2217314%22:%22#66cdaa%22%2C%2217429%22:%22#66cdaa%22%2C%2217584%22:%22#66cdaa%22%2C%2217740%22:%22#66cdaa%22%2C%2218712%22:%22#66cdaa%22%2C%2218796%22:%22#66cdaa%22%2C%2218802%22:%22#66cdaa%22%2C%2218861%22:%22#66cdaa%22%2C%2219167%22:%22#66cdaa%22%2C%2219475%22:%22#66cdaa%22%2C%2220137%22:%22#66cdaa%22%2C%2220520%22:%22#66cdaa%22%2C%2220885%22:%22#66cdaa%22%2C%2221001%22:%22#66cdaa%22%2C%2221134%22:%22#66cdaa%22%2C%2221160%22:%22#66cdaa%22%2C%2221519%22:%22#66cdaa%22%2C%2221782%22:%22#66cdaa%22%2C%2222762%22:%22#66cdaa%22%2C%2222783%22:%22#66cdaa%22%2C%2222787%22:%22#66cdaa%22%2C%2222806%22:%22#66cdaa%22%2C%2223029%22:%22#66cdaa%22%2C%2223152%22:%22#66cdaa%22%2C%2223171%22:%22#66cdaa%22%2C%2223506%22:%22#66cdaa%22%2C%2223545%22:%22#66cdaa%22%2C%2223759%22:%22#66cdaa%22%2C%2224408%22:%22#66cdaa%22%2C%2225160%22:%22#66cdaa%22%2C%2225842%22:%22#66cdaa%22%2C%2226241%22:%22#66cdaa%22%2C%2226261%22:%22#66cdaa%22%2C%2226949%22:%22#66cdaa%22%2C%2228560%22:%22#66cdaa%22%2C%2233190%22:%22#66cdaa%22%2C%2233615%22:%22#66cdaa%22%2C%22100579%22:%22#66cdaa%22%2C%22101895%22:%22#66cdaa%22%2C%22102322%22:%22#66cdaa%22%2C%22102697%22:%22#66cdaa%22%2C%22103220%22:%22#66cdaa%22%2C%22153835%22:%22#66cdaa%22%2C%22164271%22:%22#66cdaa%22%2C%22164272%22:%22#66cdaa%22%2C%22166416%22:%22#66cdaa%22%7D%2C%22name%22:%22MANC_v1.2.3_IR%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2214811%22%2C%2215524%22%2C%2216209%22%2C%2221519%22%5D%2C%22segmentQuery%22:%2214811%2C%2015524%2C%2016209%2C%2021519%22%2C%22name%22:%22MANC_v1.2.3_ir_2_04B_1%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2214098%22%2C%22164272%22%2C%2219475%22%2C%2222806%22%2C%2223506%22%2C%2226261%22%2C%2226949%22%5D%2C%22segmentQuery%22:%2214098%2C%2019475%2C%2022806%2C%2023506%2C%2026261%2C%2026949%2C%20164272%22%2C%22name%22:%22MANC_v1.2.3_ir_2_04B_2%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%22164271%22%2C%2219167%22%2C%2233615%22%5D%2C%22segmentQuery%22:%2219167%2C%2033615%2C%20164271%22%2C%22name%22:%22MANC_v1.2.3_ir_2_04B_3%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2214297%22%2C%22153835%22%5D%2C%22segmentQuery%22:%2214297%2C%20153835%22%2C%22name%22:%22MANC_v1.2.3_ir_2_04B_4%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%22100579%22%2C%22101895%22%2C%22102697%22%2C%2214886%22%2C%2218712%22%2C%2220520%22%2C%2223759%22%2C%2225842%22%2C%2226241%22%5D%2C%22segmentQuery%22:%2214886%2C%2018712%2C%2020520%2C%2023759%2C%2025842%2C%2026241%2C%20100579%2C%20101895%2C%20102697%22%2C%22name%22:%22MANC_v1.2.3_ir_2_04B_5%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22source%22%2C%22segments%22:%5B%2216938%22%2C%2218796%22%2C%2218802%22%2C%2221160%22%5D%2C%22segmentQuery%22:%2216938%2C%2018796%2C%2018802%2C%2021160%22%2C%22name%22:%22MANC_v1.2.3_ir_2_04B_6%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%22102322%22%2C%2217429%22%2C%2218861%22%5D%2C%22segmentQuery%22:%2217429%2C%2018861%2C%20102322%22%2C%22name%22:%22MANC_v1.2.3._ir_2_04B_7%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%22103220%22%5D%2C%22segmentQuery%22:%22103220%22%2C%22name%22:%22MANC_v1.2.3._ir_2_04B_8%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segmentQuery%22:%2213465%2C%2014088%2C%2014434%2C%2015369%2C14467%22%2C%22name%22:%22MANC_v1.2.3_ir_1_1%22%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2221134%22%2C%2222762%22%5D%2C%22segmentQuery%22:%2217314%2C%2017584%2C%2021134%2C%2021782%2C%2022762%2C%2022783%2C%2023171%2C%2023545%2C%2033190%2C%20166416%2C%2017740%22%2C%22name%22:%22MANC_v1.2.3_ir_1_2%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2220885%22%2C%2222787%22%2C%2223029%22%2C%2223152%22%2C%2228560%22%5D%2C%22segmentQuery%22:%2220885%2C%2022787%2C%2023029%2C%2023152%2C%2028560%22%2C%22name%22:%22MANC_v1.2.3_ir_1_3%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2221001%22%2C%2225160%22%5D%2C%22segmentQuery%22:%2221001%2C%2025160%22%2C%22name%22:%22MANC_v1.2.3_ir_1_4%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2214944%22%2C%2220137%22%2C%2224408%22%5D%2C%22segmentQuery%22:%2214944%2C%2020137%2C%2024408%22%2C%22name%22:%22MANC_v1.2.3_ir_1_5%22%2C%22visible%22:false%7D%5D%2C%22showSlices%22:false%2C%22selectedLayer%22:%7B%22flex%22:1.49%2C%22size%22:426%2C%22visible%22:true%2C%22layer%22:%22MANC_v1.2.3_ir_1_1%22%7D%2C%22crossSectionBackgroundColor%22:%22#ffffff%22%2C%22projectionBackgroundColor%22:%22#ffffff%22%2C%22layout%22:%223d%22%2C%22settingsPanel%22:%7B%22visible%22:true%7D%2C%22layerListPanel%22:%7B%22visible%22:true%7D%7D